1. 核心目标：将一个量子位（比如属于 Alice）的未知量子态，传递给相隔很远的另一个量子位（比如属于 Bob）。注意：传递的是“状态”而不是量子位这个物理实体本身，且原有的量子态在这个过程中会被破坏（符合量子不可克隆原理）。
2. 前提条件：Alice 和 Bob 之间必须提前共享一对处于纠缠态的量子位（通常是 Bell 态），并且他们之间需要有一条经典的通信通道（比如电话或互联网）。
3. 协议步骤：假设 Alice 有要传送的神秘状态 $|\psi\rangle$（量子位 0），同时 Alice 和 Bob 共享了纠缠对（量子位 1 属于 Alice，量子位 2 属于 Bob）。
* 纠缠： Alice 将她要传送的量子位 0 与她手中的纠缠对成员（量子位 1）进行相互作用（CNOT 门和 H 门）。
* 测量： Alice 对她的两个量子位（0 和 1）进行测量，得到两个经典的比特结果（00, 01, 10 或 11）。这会使 Bob 手中的量子位 2 坍缩到一个特定的状态。
* 经典通信： Alice 通过经典通道把测得的两个比特告诉 Bob。
* 状态修正： Bob 根据收到的经典信息，对他的量子位 2 施加对应的量子门（X 门、Z 门或两者都加），从而完美恢复出最初的神秘状态 $|\psi\rangle$。

In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

# 1. 设置寄存器
qr = QuantumRegister(3, name="q")
crz = ClassicalRegister(1, name="crz")
crx = ClassicalRegister(1, name="crx")
crb = ClassicalRegister(1, name="crb")
qc = QuantumCircuit(qr, crz, crx, crb)

# --- 步骤 0: 准备要隐形传态的初始状态 ---
qc.x(0)
qc.barrier()

# --- 步骤 1: Alice 和 Bob 分享纠缠对 (q1 和 q2) ---
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# --- 步骤 2: Alice 的操作 ---
qc.cx(0, 1)
qc.h(0)
qc.barrier()

# --- 步骤 3: Alice 进行测量 ---
qc.measure(0, crz)
qc.measure(1, crx)
qc.barrier()

# --- 步骤 4: Bob 根据经典信息进行状态恢复 ---
with qc.if_test((crx, 1)):
    qc.x(2)
with qc.if_test((crz, 1)):
    qc.z(2)

# --- 步骤 5: 验证测试 ---
qc.measure(2, crb)

print("隐形传态电路构建完成：")
print(qc.draw())

隐形传态电路构建完成：
       ┌───┐ ░            ░      ┌───┐ ░ ┌─┐    ░                        »
  q_0: ┤ X ├─░────────────░───■──┤ H ├─░─┤M├────░────────────────────────»
       └───┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐ ░                        »
  q_1: ──────░─┤ H ├──■───░─┤ X ├──────░──╫─┤M├─░────────────────────────»
             ░ └───┘┌─┴─┐ ░ └───┘      ░  ║ └╥┘ ░ ┌────── ┌───┐ ───────┐ »
  q_2: ──────░──────┤ X ├─░────────────░──╫──╫──░─┤ If-0  ┤ X ├  End-0 ├─»
             ░      └───┘ ░            ░  ║  ║  ░ └──╥─── └───┘ ───────┘ »
crz: 1/═══════════════════════════════════╩══╬═══════╬═══════════════════»
                                          0  ║    ┌──╨──┐                »
crx: 1/══════════════════════════════════════╩════╡ 0x1 ╞════════════════»
                                             0    └─────┘                »
crb: 1/══════════════════════════════════════════════════════════════════»
                                                                         »
«            

In [4]:
from qiskit_aer import AerSimulator

# 选择模拟器后端
simulator = AerSimulator()

# 运行电路 (采样 1000 次)
job = simulator.run(qc, shots=1000)
result = job.result()
counts = result.get_counts(qc)
f"测量统计结果: {counts}"

"测量统计结果: {'1 0 0': 265, '1 1 1': 229, '1 1 0': 279, '1 0 1': 227}"